# 기존 원판 궤적을 잇는 경계 후보 비교

검증 6명과 학습에서 본 빈 배경 2편을 같은 순서로 처리했다. 시간 연결 시에만 예외를 두자 동작 중 265개 정탐과 b06 오탐 0장을 유지하면서 b02의 77·78번이 다시 정탐이 됐다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
import pandas as pd
from matplotlib import pyplot as plt
from wlcv.evaluation.edge_continuity import OUTPUT, run

# 65번에서 원본으로부터 추론한 후보를 0번부터 다시 처리하려면 True로 바꾼다.
replay_candidates = False
scores = run() if replay_candidates else pd.read_csv(OUTPUT / "summary.csv")
details = pd.read_csv(OUTPUT / "frames.csv")
display(scores.round(3))

## 세 조건에서 달라지는 것

off는 필터 끔, strict는 경계의 납작한 후보를 항상 제외, continuity는 기존 이력으로 예상한 중심 반경 안에 있으면 허용한다. 직전 고신뢰 관측으로부터 2프레임이 지나면 예외를 적용하지 않는다. 두 모델의 이력은 따로 관리한다.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
for ax, column, title in zip(axes, ["b06_fp","a03_saved"], ["b06 false-positive frames / 94","a03 preserved recoveries / 8"]):
    bars = ax.bar(scores.rule,scores[column],color=["#777777","#227d83","#3863a0"])
    ax.set_title(title)
    ax.set_ylabel("Frames")
    ax.bar_label(bars,padding=2)
    ax.margins(y=.16)
fig.savefig(OUTPUT/"rule_counts.png",dpi=150)
plt.show()

## b02 원본에서 잘린 원판 보기

수동 박스는 초록 실선, 선택된 박스는 주황 점선이다. 두 프레임 모두 마지막 수동 이벤트 이후다. 79–91번은 수동 정답이 없어 정오 판정에서 제외한다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb

b02 = details[details.video_id.eq("b02_sn_h_02")].set_index(["rule","frame_index"])
fig, axes = plt.subplots(2,3,figsize=(9,5.7),layout="constrained")
for row, frame in enumerate([77,78]):
    image = read_rgb(VIDEOS/"b02_sn_h_02.mp4",frame)
    for col, rule in enumerate(["off","strict","continuity"]):
        ax = axes[row,col]
        ax.imshow(image)
        point = b02.loc[(rule,frame)]
        for keys,color,line in [(["gt_"+k+"_before" for k in ["x1","y1","x2","y2"]],"lime","-"),
                                (["x1","y1","x2","y2"],"darkorange","--")]:
            x,y,r,b = point[keys]
            ax.add_patch(Rectangle((x,y),r-x,b-y,fill=False,edgecolor=color,linestyle=line,linewidth=2))
        ax.set(xlim=(0,image.shape[1]),ylim=(460,0),title=f"f{frame}  {rule}  IoU {point.iou:.3f}")
        ax.axis("off")
fig.savefig(OUTPUT/"b02_edges.png",dpi=150)
plt.show()